In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

from preprocessing.geneva_stroke_unit_preprocessing.utils import create_ehr_case_identification_column
from preprocessing.geneva_stroke_unit_preprocessing.vitals_preprocessing.vitals_preprocessing import string_to_numeric


In [ ]:
data_path = '/Users/jk1/stroke_datasets/Extraction_20220815'
vitals_file_start = 'patientvalue'
restrict_to_first_72h = True

In [ ]:
vitals_files = [pd.read_csv(os.path.join(data_path, f), delimiter=';', encoding='utf-8', dtype=str)
             for f in os.listdir(data_path)
             if f.startswith(vitals_file_start)]

In [ ]:
vitals_df = pd.concat(vitals_files, ignore_index=True)

In [ ]:
vitals_df['case_admission_id'] = create_ehr_case_identification_column(vitals_df)


In [ ]:
columns_to_drop = ['nr', 'patient_id', 'eds_end_4digit', 'eds_manual', 'DOB', 'begin_date',
                   'end_date', 'death_date', 'death_hosp', 'eds_final_id',
                   'eds_final_begin', 'eds_final_end', 'eds_final_patient_id',
                   'eds_final_birth', 'eds_final_death', 'eds_final_birth_str',
                   'date_from', 'date_to', 'patient_id_manual', 'stroke_onset_date', 'Referral',
                   'match_by', 'multiple_id']
vitals_df.drop(columns_to_drop, axis=1, inplace=True)

In [ ]:
if restrict_to_first_72h:
    # keep only values from first 72h after the first value for each case_admission_id
    vitals_df['datetime'] = pd.to_datetime(vitals_df['datetime'])
    vitals_df.sort_values(['case_admission_id', 'datetime'], inplace=True)
    vitals_df['time_since_first_value'] = vitals_df.groupby('case_admission_id')['datetime'].transform(lambda x: x - x.iloc[0])
    vitals_df = vitals_df[vitals_df['time_since_first_value'] <= pd.Timedelta(hours=72)]
    vitals_df.drop('time_since_first_value', axis=1, inplace=True)

# Analyse sampling rates
sampling rate: difference between two measures for the same patient and same measurement (patient_value)
__Target parameters__: pv.ta, pv.pulse, pv.spo2, pv.fr, pv.temperature, pv.glycemia, pv.weight

output:
print: min, median, q25, q75
make plot

In [ ]:
vitals_df.patient_value.unique()

In [ ]:
vitals_df[vitals_df.patient_value == 'pv.ta']

In [ ]:
target_patient_values = [
    'pv.ta',
    'pv.pulse',
    'pv.spo2',
    'pv.fr',
    'pv.temperature',
    'pv.glycemia',
    'pv.weight',
    'lab.result.sang.p_sodium',
    'lab.result.sang.hemoglobine',
]

sampling_source = vitals_df[vitals_df['patient_value'].isin(target_patient_values)].copy()
sampling_source['datetime'] = pd.to_datetime(sampling_source['datetime'], errors='coerce')
sampling_source = sampling_source.dropna(subset=['case_admission_id', 'patient_value', 'datetime'])

# Collapse repeated rows at the same timestamp so subkeys do not inflate the sampling interval.
sampling_source = sampling_source.drop_duplicates(subset=['case_admission_id', 'patient_value', 'datetime'])
sampling_source = sampling_source.sort_values(['case_admission_id', 'patient_value', 'datetime'])
sampling_source['sampling_interval_minutes'] = (
    sampling_source.groupby(['case_admission_id', 'patient_value'])['datetime']
    .diff()
    .dt.total_seconds()
    .div(60)
)

sampling_intervals = sampling_source.dropna(subset=['sampling_interval_minutes']).copy()

summary = (
    sampling_intervals.groupby('patient_value')['sampling_interval_minutes']
    .agg(min='min', q25=lambda s: s.quantile(0.25), mean='mean', median='median', q75=lambda s: s.quantile(0.75), count='count')
    .reindex(target_patient_values)
)

print('Sampling interval between consecutive measurements for the same case and patient_value (minutes)')
print(summary.to_string(float_format=lambda x: f'{x:.2f}'))

plot_data = [
    sampling_intervals.loc[sampling_intervals['patient_value'] == patient_value, 'sampling_interval_minutes'].dropna()
    for patient_value in target_patient_values
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.boxplot(plot_data, labels=target_patient_values, showfliers=False)
ax.set_ylabel('Sampling interval (minutes)')
ax.set_title('Vital sign sampling intervals')
ax.set_yscale('log')
ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.3)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

Labs - lab.result.sang.p_sodium

In [ ]:
# then most common labs - patient value starting with lab.result.sang.
vitals_df[vitals_df.patient_value.str.startswith('lab.result.sang')].value_counts('patient_value')